In [1]:
import duckdb
con = duckdb.connect("../dbt/lakehouse.duckdb")

In [2]:
## q1
res1 = con.execute("""
    SELECT trend, COUNT(*)
    FROM household_trend
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

In [3]:
display(res1)

,trend,count_star()
0,spending_more,1410
1,spending_less,670
2,flat,396
3,lapsed,21
4,new,3


In [4]:
res = con.execute("""
select sum(spend_first_half), sum (spend_second_half)
from household_trend
""").df()

In [5]:
display(res)

,sum(spend_first_half),sum(spend_second_half)
0,3528299.15,4529144.65


## Q2 and 3: which categories are growing for the first group, and which are the second group dropping?

In [6]:
## q2:
res2 = con.execute("""
select department, share_first_half_pct, share_second_half_pct, share_change
from category_trend
where trend = 'spending_more' and spend_first_half > 1000
order by share_change desc limit 5
""").df()

In [7]:
display(res2)

,department,share_first_half_pct,share_second_half_pct,share_change
0,KIOSK-GAS,6.34,7.55,1.21
1,MISC SALES TRAN,1.48,1.73,0.25
2,DRUG GM,12.81,12.92,0.11
3,GARDEN CENTER,0.07,0.13,0.06
4,COSMETICS,0.38,0.42,0.04


In [8]:
## q3:
res3 = con.execute("""
select department, share_first_half_pct, share_second_half_pct, share_change
from category_trend
where trend = 'spending_less' and spend_first_half > 1000
order by share_change asc limit 5
""").df()

In [9]:
display(res3)

,department,share_first_half_pct,share_second_half_pct,share_change
0,GROCERY,51.57,51.00,-0.57
1,MEAT,6.99,6.46,-0.53
2,MEAT-PCKGD,5.73,5.44,-0.29
3,DELI,3.15,3.00,-0.14
4,SEAFOOD-PCKGD,0.81,0.77,-0.04


In [10]:
check2 = con.execute("""
select count(*), count(distinct department) from category_trend;
                     """).df()

In [11]:
display(check2)

,count_star(),count(DISTINCT department)
0,75,42


## Q4: demographics vs spend: which factor (demographic) affects how much a household spends?

In [12]:
res4_income = con.execute("""
select income_desc, count(*) as households, round(avg(total_spend)) as avg_spend,
round(avg(spend_per_basket)) as avg_basket
from demographic_spend group by 1 having count(*) >= 20 order by avg_spend desc
""").df()

In [13]:
display(res4_income)

,income_desc,households,avg_spend,avg_basket
0,150-174K,30,8395.0,56.0
1,125-149K,38,7912.0,48.0
2,100-124K,34,5928.0,38.0
3,75-99K,96,5823.0,42.0
4,50-74K,192,5702.0,39.0
5,Under 15K,61,5560.0,30.0
6,25-34K,77,4938.0,33.0
7,35-49K,172,4802.0,33.0
8,15-24K,74,4097.0,27.0


In [14]:
res4_housesize = con.execute("""
select household_size_desc, count(*) as households, round(avg(total_spend)) as avg_spend,
round(avg(spend_per_basket)) as avg_basket
from demographic_spend group by 1 having count(*) >= 20 order by avg_spend desc
""").df()

In [15]:
display(res4_housesize)

,household_size_desc,households,avg_spend,avg_basket
0,5+,66,6699.0,44.0
1,4,53,6613.0,44.0
2,3,109,6060.0,40.0
3,2,318,5538.0,38.0
4,1,255,5032.0,31.0


In [16]:
res4_kidCategory = con.execute("""
select kid_category_desc, count(*) as households, round(avg(total_spend)) as avg_spend,
round(avg(spend_per_basket)) as avg_basket
from demographic_spend group by 1 having count(*) >= 20 order by avg_spend desc
""").df()

In [17]:
display(res4_kidCategory)

,kid_category_desc,households,avg_spend,avg_basket
0,2,60,6764.0,45.0
1,3+,69,6742.0,43.0
2,1,114,5875.0,39.0
3,None/Unknown,558,5299.0,35.0


In [18]:
res4_age_desc = con.execute("""
select age_desc, count(*) as households, round(avg(total_spend)) as avg_spend,
round(avg(spend_per_basket)) as avg_basket
from demographic_spend group by 1 having count(*) >= 20 order by avg_spend desc
""").df()

In [19]:
display(res4_age_desc)

,age_desc,households,avg_spend,avg_basket
0,35-44,194,6403.0,40.0
1,45-54,288,5755.0,37.0
2,25-34,142,5478.0,39.0
3,55-64,59,5055.0,33.0
4,19-24,46,4708.0,29.0
5,65+,72,4241.0,33.0


## Q5: do campaigns improve engagement?

In [20]:
res5 = con.execute("""
select count(*) as pairs, sum(baskets_before) as before, sum(baskets_during) as during,
round(sum(baskets_during) * 1.0 / sum(baskets_before), 2) as ratio
from campaign_effect
""").df()

In [21]:
display(res5)

,pairs,before,during,ratio
0,7208,99259.0,96568.0,0.97


In [22]:
exit()